# Model Selection and Evaluation - Part 2

## Activity 3 - Cross Validation

In the previous activity, we made the classic mistake of using our validation data to predict our generalization error. This tends to give misleadingly optimistic predictions about how well we will do on unobserved data. Remember that we carefully picked our hyperparameter values to do as well as possible *on our held-out data*. We shouldn't be surprised when our model performs better on that data than on unobserved data. This problem is particularly acute if our data set is small.

The traditional solution is to divide our data into three disjoint sets: **training**, **validation**, and **testing**:
* The **training** set is used to fit the model.
* The **validation** set is used to evaluate models for the purpose of hyperparameter selection. 
* The **test** set is kept in a locked room guarded by jaguars. We only look at the testing set ONCE, when we have finalized our model. That way our performance on the test set gives us an unbiased estimate of our generalization error.

This traditional approach is fine if we have a lot of data to work with. If the data set is small, we are faced with a painful dilemma: More validation data means better model selection. More testing data means more accurate model evaluation. More training data means better models. Any data we use for one purpose can't be used for the others.

**Cross validation** is one way to use limited data more effectively.  The cells below walk us through an example of using cross validation for hyperparameter tuning.

In [ ]:
# We need to reimport and reload everything...
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
import datasource

from sklearn.tree import DecisionTreeRegressor

# Grab our training data
source = datasource.DataSource()
X, y = source.gen_data(100, seed=100)

# Split our data into a training and testing set...
split_point = int(X.shape[0] * .8) # Use 80% of the data to train the model

X_train = X[0:split_point, :]
y_train = y[0:split_point]

X_test = X[split_point::, :] # This data will ONLY be used for final evaluation.
y_test = y[split_point::]

The following cell shows how we can use the scikit-learn `KFold` class to automatically split up our training data for k-fold cross validation. Take a minute to read through this code to make sure you understand what's going on.

In [ ]:
from sklearn.model_selection import KFold

folds = 10
max_max_leaves = 80
kf = KFold(n_splits=folds)

mses = np.zeros((folds, max_max_leaves - 2)) # (can't have 0 or 1 leaves)

# Loop over all of the hyperparameter settings
for max_leaves in range(2, max_max_leaves):
    
    k = 0
    
    # Evaluate each one K-times
    for train_index, val_index in kf.split(X_train):
        X_tr, X_val = X_train[train_index], X_train[val_index]
        y_tr, y_val = y_train[train_index], y_train[val_index]

        tree = DecisionTreeRegressor(max_leaf_nodes=max_leaves)
        tree.fit(X_tr, y_tr)
        
        y_val_predict = tree.predict(X_val)
        mses[k, max_leaves - 2] = np.sum((y_val - y_val_predict)**2) / y_val.size
        
        k += 1
        
# Average across the k folds
mse_avg = np.mean(mses, axis=0)

plt.plot(np.arange(2, max_max_leaves), mse_avg)
plt.xlabel('max leaves')
plt.ylabel('MSE')
plt.show()

If we are real experts in scikit-learn, we can automate some of this by using the `cross_val_score` function:  

(There are also library routines for [automating the entire process of hyperparameter tuning](https://scikit-learn.org/stable/modules/grid_search.html).) 

In [ ]:
from sklearn.model_selection import cross_val_score

mses = np.zeros((folds,max_max_leaves - 2))

# Loop over all of the hyperparameter settings
for size in range(2, max_max_leaves):
    tree = DecisionTreeRegressor(max_leaf_nodes=size)
    
    # Returns an array of cross validation results.
    mses[:, size - 2] = -cross_val_score(tree, X_train, y_train, 
                                         cv=folds, scoring='neg_mean_squared_error')
    
mse_avg = np.mean(mses, axis=0)

plt.plot(np.arange(2, 80), mse_avg)
plt.show()
plt.xlabel('max leaves')
plt.ylabel('MSE')


### Question

* Based on the results above, what is the most promising hyperparameter value?

### Answer

* 

Now that we have a value for our hyperparameter, let's train our final model on the *full* training set and use our locked-away testing set to predict model performance.

In [ ]:
tree = DecisionTreeRegressor(max_leaf_nodes=????) # Put your best hyperparameter here!

# Train using ALL the training data
tree.fit(X_train, y_train)

# Test on held-out testing data
y_test_predict = tree.predict(X_test)
mse = np.sum((y_test - y_test_predict)**2) / y_test.size

print("Predicted MSE: {:.4f}".format(mse))

Since none of the data we are testing on here was used *in any way* to design or fit the model, this value should give us an unbiased estimate of our generalization error.  Let's try testing on some new unobserved data to see how good our estimate is:

In [ ]:

# Let's see how we do on unobserved data... 
X_new, y_new = source.gen_data(5000, seed=200)
y_new_predict = tree.predict(X_new)
mse = np.sum((y_new - y_new_predict)**2) / y_new.size
print("MSE: {:.4f}".format(mse))

### Questions

Take a look back at the results from Exercise 2 and answer the following questions:

* Did the cross validation approach improve our results in terms of model selection (i.e. did we end up with a better model)?  Justify your answer.
* Did maintaining a proper test set improve our results in terms of model evaluation (i.e. did we make a more accurate prediction about our generalization error)?  Justify your answer.

### Answers

* 
* 

## Exercise 4 — Understanding Generalization Error

So far, we have focused on the idea of **generalization**: a model should perform well not only on the training data, but also on new data that it has not seen before.

One way to think about a model's prediction error is to separate it into three sources:

- **Bias:** error caused by the model being too simple or making overly restrictive assumptions.
- **Variance:** error caused by the model being too sensitive to the particular training set it was given.
- **Irreducible error:** randomness or noise in the data that cannot be eliminated by changing the model.

This idea is sometimes summarized by the **bias–variance decomposition**:

$$
E\left[(y-\hat{f}(x))^2\right]
=
\left(\operatorname{Bias}[\hat{f}(x)]\right)^2
+
\operatorname{Variance}[\hat{f}(x)]
+
\sigma^2
$$

You do not need to derive this equation. Instead, use the ideas above to answer the following questions.

### Questions

* What part of the formula above corresponds to the notion of *generalization error* that we have been discussing?
* How might changing a model's hyperparameters affect its bias? For example, what might happen if we make a model more or less flexible?
* What would you expect to happen to a model's variance as the size of the training set increases? Explain why.
* Suppose a model performs extremely well on its training set but poorly on new data. Which component of the decomposition is most closely related to this behavior, and why?


### Answers

* 
* 
*
* 

## Exercise 5 - Nested Cross Validation

 There is still one limitation to our approach from the previous exercise: **our testing set was relatively small**, which can make our estimate of model performance less reliable. For example, the test set might happen to contain particularly difficult—or particularly easy—instances. In that case, our estimate of the model's generalization error could be misleading.

 Ideally, we would like to make use of **all of our data** for training and model tuning while also evaluating the model on data that was not used during training or tuning. **Nested cross-validation** provides a way to accomplish this.

 Nested cross-validation uses **two levels of K-fold cross-validation**:

 1. **Outer cross-validation** evaluates the model and estimates its generalization error.
2. **Inner cross-validation** tunes the model's hyperparameters using only the training data from the outer loop.

 The important idea is that the outer test data remains completely separate from the model-tuning process.

 ### How Nested Cross-Validation Works

 At the outer level, we repeatedly divide the dataset into **training and testing sets**. For each outer split, the training portion is then passed to an inner cross-validation procedure.

 The inner cross-validation is used to select the best hyperparameters. Once the best hyperparameters have been chosen, the model is trained using the entire outer training set and then evaluated on the outer test set.

```
Entire dataset
│
├── Outer Fold 1
│   ├── Test data       ← used only for final evaluation
│   └── Training data
│       └── Inner CV    ← choose hyperparameters
│
├── Outer Fold 2
│   ├── Test data       ← used only for final evaluation
│   └── Training data
│       └── Inner CV    ← choose hyperparameters
│
├── Outer Fold 3
│   ├── Test data       ← used only for final evaluation
│   └── Training data
│       └── Inner CV    ← choose hyperparameters
│
└── ...
```

 For example, suppose we use **5-fold cross-validation for the outer loop** and **3-fold cross-validation for the inner loop**.

```
Outer Fold 1:
    Outer Training Data
        ↓
    Inner 3-fold CV
        ↓
    Choose hyperparameters
        ↓
    Train final model
        ↓
    Outer Test Data
        ↓
    Evaluate model

Outer Fold 2:
    Outer Training Data
        ↓
    Inner 3-fold CV
        ↓
    Choose hyperparameters
        ↓
    Train final model
        ↓
    Outer Test Data
        ↓
    Evaluate model

...
```

 The process is repeated for each outer fold. In a 5-fold outer cross-validation, this gives us **five independent estimates of model performance**, which can then be summarized, for example, by taking their mean.

 ### Why Is This Useful?

 Suppose we are training a decision tree and need to choose an appropriate value for `max_depth`:

```
max_depth = 2, 3, 4, 5, ...
```

 We could use cross-validation to determine which value works best. However, if we use the same data both to **select the hyperparameters** and to **evaluate the final model**, our estimate of performance can be overly optimistic.

 Nested cross-validation avoids this problem by separating **model tuning** from **model evaluation**:

```
Outer Training Data
        ↓
    Inner CV
        ↓
Choose hyperparameters
        ↓
Train model
        ↓
Outer Test Data
        ↓
Estimate generalization error
```

 The key point is that the **outer test fold is never used to select the hyperparameters**. It is held out until the very end of each outer iteration and is used only to evaluate the resulting model.

 This separation helps prevent **data leakage** and gives us a more reliable estimate of how the complete model-selection process will perform on unseen data.

 ### Nested Cross-Validation in scikit-learn

 In `scikit-learn`, we can implement nested cross-validation by placing a `GridSearchCV` inside `cross_val_score`.

```python
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score

model = DecisionTreeClassifier(random_state=42)

param_grid = {
    "max_depth": [1, 2, 3, 4, 5],
    "min_samples_split": [2, 3, 4]
}

# Inner CV: select the best hyperparameters
inner_cv = KFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

# Outer CV: evaluate the model-selection process
outer_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=inner_cv
)

scores = cross_val_score(
    grid,
    X,
    y,
    cv=outer_cv
)

print("Scores:", scores)
print("Mean:", scores.mean())
```

 Here, `GridSearchCV` performs the **inner cross-validation** and selects the best hyperparameters. The `cross_val_score` function performs the **outer cross-validation**, using each outer test fold to evaluate the resulting model.

 
 For an example of nested cross-validation using `scikit-learn`, see the [scikit-learn documentation](<https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html>).

### Try It Yourself

 If you have time, use nested cross-validation to redo the example from the previous exercise.

 Compare the resulting estimate of generalization error with the estimate you obtained previously.

 **Does nested cross-validation give you a different or more reliable estimate of generalization error? Why might that be the case?**


### Key Idea

 > **The inner CV chooses the model; the outer CV evaluates the model-selection process.**

 The inner loop answers **"Which model or hyperparameters should I use?"**, while the outer loop answers **"How well does this entire model-selection procedure generalize to unseen data?"**
